Here we impute (with zeros) the Watershed-SV output before running the Watershed algorithm.

For demonstration purposes and due to our limited sample size, we use the same data for both training and testing.

## Setup

In [1]:
!pip install pysam


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pysam import VariantFile
import pandas as pd
from firecloud import fiss
import functools
import io
import os
import numpy as np
import argparse

Imputation functions:

In [3]:
def drop_uninformative_columns(dataframe,mode='evaluate'):
    to_drop=[]
    if mode=='evaluate':
        training=dataframe[dataframe.N2pair=="NA"]
    elif mode=='predict':
        training=dataframe
    for i in training.columns:
        if i!="N2pair" and i!="partition" and training[i].value_counts().max()==training.shape[0]:
            to_drop.append(i)
    return dataframe.drop(to_drop,axis=1),to_drop

def run(training, testings=[], testing_list='None', min_af_impute_mode="infer", min_af_value=1/831, out_prefix="out_prefix"):
        training=pd.read_csv(training,sep='\t')
        test_dfs = []
        if testing_list != 'None':
            with open(testing_list,'r') as handle:
                for i in handle.readlines():
                    test_dfs.append(pd.read_csv(i.rstrip(),sep='\t'))
        test_dfs.append(pd.read_csv(testings,sep='\t')) # ADDED
        testing=pd.concat(test_dfs)
        if 'SVTYPE_INV' in testing:
            testing.loc[testing['SVTYPE_INV']==1,'SVTYPE_DEL'] = 1
            testing.loc[testing['SVTYPE_INV']==1,'SVTYPE_INS'] = 1
        if 'SVTYPE_MEI' in testing:
            testing.loc[testing['SVTYPE_MEI']==1,'SVTYPE_DEL'] = 1
        # add partition marker
        training['partition'] = 'train'
        testing['partition'] = 'test'
        to_drop = list(set(testing.columns).difference(set(training.columns)))
        total_data = pd.concat([training,testing]).drop(to_drop,axis=1)
        outlier_cols = total_data.columns[total_data.columns.str.contains('TE_pvalues')].to_list()
        total_data['N2pair'] = 'NA'
        total_data[outlier_cols] = total_data[outlier_cols].fillna('NaN')
        total_data.af.fillna(0,inplace=True)
        # impute af=0
        if min_af_impute_mode == 'upload':
            total_data['af'] = np.where(total_data['af']==0, min_af_value, total_data['af'])
        else:
            inferred_min = total_data[total_data['af']!=0].af.min()
            total_data['af'] = np.where(total_data['af']==0,inferred_min,total_data['af'])
        # impute all the rest
        methods=pd.read_csv('collapse_annotation_instructions.tsv',sep='\t')
        impute_records=methods[~methods.Impute_method.isna()].to_records()
        impute_dict={record[1]:record[2] for record in impute_records}
        for column in total_data.columns:
            if column in impute_dict:
                if impute_dict[column]=='0':
                    total_data[column]=total_data[column].fillna(0)
                elif impute_dict[column]=='mean':
                    impute_val = total_data.loc[total_data.N2pair=='NA',column].mean()
                    total_data[column]=total_data[column].fillna(impute_val)
        total_data.fillna(0,inplace=True)
        # then drop the uninformative columns from the data. 
        _,drop_cols = drop_uninformative_columns(total_data[total_data.partition=='train'],mode='predict')
        total_data[total_data.partition=="train"][[c for c in total_data if c not in drop_cols +['partition', 'N2pair']+outlier_cols] + outlier_cols + ['N2pair']].\
        to_csv(f'{out_prefix}_training_data.tsv',sep='\t',header=True,index=False)
        total_data[total_data.partition=="test"][[c for c in total_data if c not in drop_cols+['partition','N2pair']+outlier_cols] + outlier_cols + ['N2pair']].\
        to_csv(f'{out_prefix}_testing_data.tsv',sep='\t',header=True,index=False)

The imputation function depends on the `collapse_annotation_instructions.tsv` file, which defines which function should be used to collapse an annotation.

In [4]:
tbl = pd.read_csv(io.StringIO(fiss.fapi.get_entities_tsv(
  namespace = os.environ['WORKSPACE_NAMESPACE'],
  workspace = os.environ['WORKSPACE_NAME'],
  etype = 'anno_metadata',
  model='flexible').text), sep='\t')

!gcloud storage cp "$WORKSPACE_BUCKET/data/derived/combined_dataset-400_random_pairs.tsv" .
!gcloud storage cp "{tbl.loc[tbl['entity:anno_metadata_id']=='Collapse_instructions', 'file'].iloc[0]}" .

Copying gs://fc-secure-ca459dc2-db2a-4dbb-8747-baea7e6c6669/data/derived/combined_dataset-400_random_pairs.tsv to file://./combined_dataset-400_random_pairs.tsv
  Completed files 1/1 | 79.2kiB/79.2kiB                                        
Copying gs://fc-secure-ca459dc2-db2a-4dbb-8747-baea7e6c6669/data/raw/annotation/collapse_annotation_instructions.tsv to file://./collapse_annotation_instructions.tsv
  Completed files 1/1 | 4.7kiB/4.7kiB                                          


## Impute

In [5]:
run(training = 'combined_dataset-400_random_pairs.tsv',
    testings = 'combined_dataset-400_random_pairs.tsv',
    min_af_impute_mode = 'infer',
    out_prefix = 'subset')
run(training = 'combined_dataset.tsv',
    testings = 'combined_dataset.tsv',
    min_af_impute_mode = 'infer',
    out_prefix = 'full')

## Upload

In [6]:
os.listdir()
!gcloud storage cp "subset_training_data.tsv" "$WORKSPACE_BUCKET/data/derived/"
!gcloud storage cp "subset_testing_data.tsv"  "$WORKSPACE_BUCKET/data/derived/"
!gcloud storage cp   "full_training_data.tsv" "$WORKSPACE_BUCKET/data/derived/"
!gcloud storage cp   "full_testing_data.tsv"  "$WORKSPACE_BUCKET/data/derived/"

Copying file://subset_training_data.tsv to gs://fc-secure-ca459dc2-db2a-4dbb-8747-baea7e6c6669/data/derived/subset_training_data.tsv
  Completed files 1/1 | 95.2kiB/95.2kiB                                        
Copying file://subset_testing_data.tsv to gs://fc-secure-ca459dc2-db2a-4dbb-8747-baea7e6c6669/data/derived/subset_testing_data.tsv
  Completed files 1/1 | 95.2kiB/95.2kiB                                        
Copying file://full_training_data.tsv to gs://fc-secure-ca459dc2-db2a-4dbb-8747-baea7e6c6669/data/derived/full_training_data.tsv
  Completed files 1/1 | 123.5kiB/123.5kiB                                      
Copying file://full_testing_data.tsv to gs://fc-secure-ca459dc2-db2a-4dbb-8747-baea7e6c6669/data/derived/full_testing_data.tsv
  Completed files 1/1 | 123.5kiB/123.5kiB                                      
